Celda 1 — Setup

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

silver_table = f"{catalog}.{schema}.silver_orders"
dim_customer_table = f"{catalog}.{schema}.dim_customer"
dim_product_table = f"{catalog}.{schema}.dim_product"
dim_location_table = f"{catalog}.{schema}.dim_location"
dim_date_table = f"{catalog}.{schema}.dim_date"
fact_shipments_table = f"{catalog}.{schema}.fact_shipments"

df_silver = spark.table(silver_table)

print(f"Silver: {silver_table} | Filas: {df_silver.count()}")

Celda 2 — dim_customer

In [0]:
from pyspark.sql.functions import col

df_dim_customer = (
    df_silver
    .select(
        "Customer_Id",
        "Customer_Fname",
        "Customer_Lname",
        "Customer_Email",
        "Customer_Segment",
        "Customer_City",
        "Customer_State",
        "Customer_Country",
        "Customer_Street",
        "Customer_Zipcode"
    )
    .dropDuplicates(["Customer_Id"])
)

dim_customer_table = f"{catalog}.{schema}.dim_customer"

(
    df_dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dim_customer_table)
)

print(f"dim_customer creada: {spark.table(dim_customer_table).count()} clientes únicos")

Celda 3 — dim_product

In [0]:
df_dim_product = (
    df_silver
    .select(
        "Product_Card_Id",
        "Product_Name",
        "Product_Description",
        "Product_Price",
        "Product_Status",
        "Category_Id",
        "Category_Name",
        "Department_Id",
        "Department_Name"
    )
    .dropDuplicates(["Product_Card_Id"])
)

dim_product_table = f"{catalog}.{schema}.dim_product"

(
    df_dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dim_product_table)
)

print(f"dim_product creada: {spark.table(dim_product_table).count()} productos únicos")

Celda 4 — dim_location

In [0]:
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit

location_key_cols = ["Order_City", "Order_State", "Order_Country", "Order_Region", "Market", "Order_Zipcode"]

def build_location_key(df):
    return sha2(
        concat_ws("||", *[coalesce(col(c).cast("string"), lit("UNKNOWN")) for c in location_key_cols]),
        256
    )

df_dim_location = (
    df_silver
    .select(*location_key_cols)
    .dropDuplicates()
    .withColumn("location_id", build_location_key(df_silver))
)

(
    df_dim_location.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dim_location_table)
)

print(f"dim_location creada: {spark.table(dim_location_table).count()} ubicaciones únicas")

Celda 5 — dim_date

Vamos a construir un calendario cubriendo el rango completo de fechas que aparecen en el dataset (tanto order_date como shipping_date), no solo una de las dos — así la dimensión sirve para analizar por cualquiera de los dos eventos.

In [0]:
from pyspark.sql.functions import explode, sequence, to_date, min as spark_min, max as spark_max, year, month, dayofmonth, dayofweek, date_format, quarter

date_range = df_silver.select(
    spark_min(to_date("order_date_DateOrders")).alias("min_date"),
    spark_max(to_date("shipping_date_DateOrders")).alias("max_date")
).collect()[0]

df_dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{date_range['min_date']}'), to_date('{date_range['max_date']}'), interval 1 day)) as full_date")
    .withColumn("date_id", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year("full_date"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("day_of_week", dayofweek("full_date"))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
)

dim_date_table = f"{catalog}.{schema}.dim_date"

(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dim_date_table)
)

print(f"dim_date creada: {spark.table(dim_date_table).count()} fechas")
print(f"Rango: {date_range['min_date']} a {date_range['max_date']}")

Celda 6 — fact_shipments

Aquí conectamos Silver con las dimensiones que acabamos de crear, usando joins para traer las llaves foráneas correctas (especialmente location_id, que es un surrogate key generado, así que debemos "pegarlo" de vuelta vía join por las columnas naturales).

In [0]:
df_fact = (
    df_silver
    .withColumn("location_id", build_location_key(df_silver))
    .withColumn("order_date_id", date_format(col("order_date_DateOrders"), "yyyyMMdd").cast("int"))
    .withColumn("shipping_date_id", date_format(col("shipping_date_DateOrders"), "yyyyMMdd").cast("int"))
    .select(
        "Order_Item_Id", "Order_Id",
        col("Customer_Id").alias("customer_id"),
        col("Product_Card_Id").alias("product_id"),
        "location_id", "order_date_id", "shipping_date_id",
        "Days_for_shipping_real", "Days_for_shipment_scheduled",
        "is_late_delivery", "Late_delivery_risk", "Shipping_Mode",
        "Order_Item_Quantity", "Order_Item_Product_Price",
        "Order_Item_Discount", "Order_Item_Discount_Rate",
        "Sales", "Order_Item_Total", "Benefit_per_order",
        "Order_Profit_Per_Order", "Order_Item_Profit_Ratio",
        "Order_Status", "Delivery_Status"
    )
)

(
    df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(fact_shipments_table)
)

print(f"fact_shipments creada: {spark.table(fact_shipments_table).count()} filas")

nulls_location = spark.table(fact_shipments_table).filter(col("location_id").isNull()).count()
print(f"Filas sin location_id: {nulls_location}")

Celda 7 — KPIs de OTIF y rentabilidad

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, count, round as spark_round

df_fact = spark.table(fact_shipments_table)

# OTIF global
otif = (
    df_fact
    .agg(
        spark_round((1 - (spark_sum(col("is_late_delivery").cast("int")) / count("*"))) * 100, 2).alias("otif_pct")
    )
)
display(otif)

# OTIF y retraso promedio por modo de envío
otif_by_mode = (
    df_fact
    .groupBy("Shipping_Mode")
    .agg(
        count("*").alias("total_shipments"),
        spark_round((1 - (spark_sum(col("is_late_delivery").cast("int")) / count("*"))) * 100, 2).alias("otif_pct"),
        spark_round(avg(col("Days_for_shipping_real") - col("Days_for_shipment_scheduled")), 2).alias("avg_delay_days")
    )
    .orderBy("Shipping_Mode")
)
display(otif_by_mode)

# Rentabilidad por categoría de producto
profit_by_category = (
    df_fact.join(spark.table(dim_product_table), df_fact.product_id == col("Product_Card_Id"), "left")
    .groupBy("Category_Name")
    .agg(
        spark_round(spark_sum("Sales"), 2).alias("total_sales"),
        spark_round(spark_sum("Benefit_per_order"), 2).alias("total_profit"),
        spark_round(spark_sum("Benefit_per_order") / spark_sum("Sales") * 100, 2).alias("margin_pct")
    )
    .orderBy(col("total_sales").desc())
)
display(profit_by_category)

Celda 8 — Tablas Gold de KPIs

In [0]:
otif_by_mode_table = f"{catalog}.{schema}.gold_otif_by_shipping_mode"
profit_by_category_table = f"{catalog}.{schema}.gold_profit_by_category"

(
    otif_by_mode.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(otif_by_mode_table)
)

(
    profit_by_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(profit_by_category_table)
)

print(f"Creadas: {otif_by_mode_table}, {profit_by_category_table}")
